CREAMOS TABLAS DELTA

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.electiva.customers AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos/customers.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE workspace.electiva.employee AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos/employees.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE workspace.electiva.orders AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos/orders.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE workspace.electiva.product_categories AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos/productcategories.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE workspace.electiva.product_sub_categories AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos/productsubcategories.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE workspace.electiva.products AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos/products.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE workspace.electiva.vendor AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos/vendors.csv', format => 'csv', header => true);

CREATE OR REPLACE TABLE workspace.electiva.vendor_product AS
SELECT * FROM read_files('/Volumes/workspace/electiva/datos/vendorproduct.csv', format => 'csv', header => true);

In [0]:
%sql
SELECT * FROM workspace.electiva.customers LIMIT 5;

PUNTO 1

In [0]:
%sql
SELECT 
    customerid,
    CONCAT(firstname, ' ', lastname) AS fullname,
    CONCAT(customerid, '-', UPPER(LEFT(firstname, 2)), '-', LOWER(LEFT(lastname, 2)), '-AW') AS user_name
FROM workspace.electiva.customers;

PUNTO 2

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.electiva.page_user (
    customerid INT,
    user_name STRING
);

INSERT INTO workspace.electiva.page_user
SELECT 
    customerid,
    CONCAT(customerid, '-', UPPER(LEFT(firstname, 2)), '-', LOWER(LEFT(lastname, 2)), '-AW')
FROM workspace.electiva.customers;

SELECT * FROM workspace.electiva.page_user;

PUNTO 3
Creamos una vista que filra ordenes de Agosto del 2011 y asigna el descuento por categoría (Bikes 10 %, Clothing 15 %, Components 8 %, Accessories 20 % y 0 para las demás)

In [0]:
%sql
SELECT table_name, column_name
FROM workspace.information_schema.columns
WHERE table_schema = 'electiva'
  AND table_name IN ('orders', 'products', 'product_sub_categories', 'product_categories')
ORDER BY table_name, ordinal_position;

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.electiva.v_ordenes_mitad_anio AS
WITH ordenes_agosto AS (
    SELECT 
        p.productname,
        p.modelname,
        u.user_name,
        o.salesorderid,
        c.categoryname,
        o.orderdate,
        o.linetotal,
        CASE 
            WHEN c.categoryname ILIKE 'Bikes' THEN 0.10
            WHEN c.categoryname ILIKE 'Clothing' THEN 0.15
            WHEN c.categoryname ILIKE 'Components' THEN 0.08
            WHEN c.categoryname ILIKE 'Accessories' THEN 0.20
            ELSE 0
        END AS `descuento_mitad_año`
    FROM workspace.electiva.orders o
    JOIN workspace.electiva.page_user u ON o.customerid = u.customerid
    JOIN workspace.electiva.products p ON o.productid = p.productid
    JOIN workspace.electiva.product_sub_categories s ON p.subcategoryid = s.subcategoryid
    JOIN workspace.electiva.product_categories c ON s.categoryid = c.categoryid
    WHERE o.orderdate >= '2011-08-01' AND o.orderdate < '2011-09-01'
)
SELECT *, linetotal - linetotal * `descuento_mitad_año` AS total_descuentos_aplicados
FROM ordenes_agosto;

SELECT * FROM workspace.electiva.v_ordenes_mitad_anio;

PUNTO 4  
Creamos una vista materializada que filtra las órdenes de 2011 y agrupa por cliente.

In [0]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW workspace.electiva.customer_class AS
SELECT 
    COUNT(DISTINCT o.salesorderid) AS amount_of_orders,
    u.user_name,
    c.fullname AS customerfullname,
    CASE 
        WHEN COUNT(DISTINCT o.salesorderid) >= 3 THEN 'Oro'
        WHEN COUNT(DISTINCT o.salesorderid) = 2 THEN 'Plata'
        ELSE 'Ocasional'
    END AS user_level,
    c.customerid
FROM workspace.electiva.orders o
JOIN workspace.electiva.customers c ON o.customerid = c.customerid
JOIN workspace.electiva.page_user u ON c.customerid = u.customerid
WHERE YEAR(o.orderdate) = 2011
GROUP BY c.customerid, c.fullname, u.user_name;

SELECT * FROM workspace.electiva.customer_class;